In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [ ]:
import numpy as np
from functools import partial

import models.merton as merton

import sims.gbm as gbm
import sims.engine as mce

# Parameters
V0 = 100
D1 = 50
D2 = 50
r = 0.01
sigma = 0.25
T1 = 1.0
T2 = 2.0

times = np.linspace(0, T1, 2)
n_paths = 100_000

events = {
    T1: {
        "equity_value_wo_asset_sale": lambda V: np.maximum(merton.equity_value(V, D2, sigma, r, T2 - T1) - D1, 0),
        "first_cpn_wo_asset_sale": lambda V: np.where(merton.equity_value(V, D2, sigma, r, T2 - T1) > D1, D1, (D1 / (D1 + D2)) * V),
        "equity_value_w_asset_sale": lambda V: merton.equity_value(np.maximum(V - D1, 1e-6), D2, sigma, r, T2 - T1),
        "first_cpn_w_asset_sale": lambda V: np.where(V >= D1, D1, (D1 / (D1+D2)) * V),
    }
}

results = mce.run_monte_carlo(
    initial_state_fn=partial(gbm.initial_state, S0=V0),
    step_fn=partial(gbm.step, mu=r, sigma=sigma),
    times=times,
    n_paths=n_paths,
    events=events,
    seed=42,
)
results

S_wo_asset_sale = np.mean(results["equity_value_wo_asset_sale"][0]) * np.exp(-r*T1)
S_w_asset_sale  = np.mean(results["equity_value_w_asset_sale"][0]) * np.exp(-r*T1)
B_wo_asset_sale = V0 - S_wo_asset_sale
B_w_asset_sale  = V0 - S_w_asset_sale
B1_wo_asset_sale = np.mean(results["first_cpn_wo_asset_sale"][0]) * np.exp(-r*T1)
B1_w_asset_sale  = np.mean(results["first_cpn_w_asset_sale"][0]) * np.exp(-r*T1)
B2_wo_asset_sale = B_wo_asset_sale - B1_wo_asset_sale
B2_w_asset_sale  = B_w_asset_sale - B1_w_asset_sale
# put_mc = np.mean(results["put_payoff"][0]) * np.exp(-r*T)

print("Equity value w/o asset sale:", S_wo_asset_sale)
print("Equity value w asset sale:", S_w_asset_sale)


Equity value w/o asset sale: 10.617007691274097
Equity value w asset sale: 11.762313232809571


In [20]:
times = np.linspace(0, T, 2)
times

array([0., 1.])

In [ ]:
from tabulate import tabulate

# Data
data = [
    ["Equity value", S_wo_asset_sale, S_w_asset_sale],
    ["Debt value", B_wo_asset_sale, B_w_asset_sale],
    ["First coupon value", B1_wo_asset_sale, B1_w_asset_sale],
]

headers = ["", "Asset sales not allowed", "Asset sales allowed"]

print(tabulate(data, headers=headers, tablefmt="fancy_grid"))

╒════════════════════╤═══════════════════════════╤═══════════════════════╕
│                    │   Asset sales not allowed │   Asset sales allowed │
╞════════════════════╪═══════════════════════════╪═══════════════════════╡
│ Equity value       │                   10.617  │               11.7623 │
├────────────────────┼───────────────────────────┼───────────────────────┤
│ Debt value         │                   89.383  │               88.2377 │
├────────────────────┼───────────────────────────┼───────────────────────┤
│ First coupon value │                   44.7678 │               49.3987 │
╘════════════════════╧═══════════════════════════╧═══════════════════════╛
